# PDF Question Answering System using RAG

## Objective

Build a system that:

1. Reads a PDF
2. Stores document chunks in FAISS
3. Retrieves relevant content
4. Answers questions using GPT-4o-mini

In [4]:
# !pip install pypdf

In [ ]:
# !pip install faiss-cpu

  Using cached faiss_cpu-1.14.2-cp311-cp311-win_amd64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.14.2-cp311-cp311-win_amd64.whl (16.1 MB)


In [ ]:
# !pip install sentence-transformers

  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.22.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached sentence_transformers-5.5.1-py3-none-any.whl (588 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------------- ---------------------- 3.7/8.3 MB 24.2 MB/s eta 0:00:01
   ----------------- ---------------------- 3.7/8.3 MB 24.2 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.3 MB 10.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.3 MB 8.2 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.3 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 6.8 MB/s  0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cache

In [ ]:
# !pip install openai

In [ ]:
# !pip install python-dotenv

In [7]:
import os
import faiss
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

from dotenv import load_dotenv
from openai import OpenAI

In [8]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

## Step 1: Extract PDF Text

In [10]:
PDF_PATH = r"C:\Users\deves\OneDrive\Documents\Gen AI Assignments\Week 3 - PDF-RAG\AWL Limited Annual Report for FY2024-25.pdf"

In [11]:
reader = PdfReader(PDF_PATH)

full_text = ""

for page in reader.pages:
    full_text += page.extract_text()

In [12]:
print(full_text[:1000])

 
 
AWL Agri Business Ltd. 
Formerly known as Adani Wilmar Ltd. 
Fortune House 
Nr Navrangpura Railway Crossing, 
Ahmedabad 380 009, Gujarat, India  
CIN: L15146GJ1999PLC035320 
Tel +91 79 2645 5650 
Fax +91 79 2645 5621 
info@awl.in 
www.awl.in  
Registered Oﬃce: Fortune House, Nr Navrangpura Railway Crossing, Ahmedabad 380 009, Gujarat, India  
 
Ref No: AWL/SECT/2025-26/20 
 
June 3, 2025 
 
BSE Limited 
Floor 25, P J Towers, 
Dalal Street, 
Mumbai – 400 001 
National Stock Exchange of India Limited 
Exchange Plaza,  
Bandra Kurla Complex, 
Bandra (E), Mumbai – 400 051 
Scrip Code: 543458 Scrip Code: AWL 
 
Sub: Annual General Meeting Notice and Annual Report for the FY 2024-25 of Adani Wilmar Limited 
(“the Company”). 
 
Dear Sir/Madam, 
 
This is to inform that the 27 th Annual General Meeting (“AGM”) of the Company will be held on 
Thursday, 26th June, 2025 at 11.00 AM (IST) through Video Conferencing / Other Audio Visual Means 
in accordance with the applicable circulars issued 

## Step 2: Create Chunks

In [13]:
chunk_size = 1000

chunks = [
    full_text[i:i+chunk_size]
    for i in range(0, len(full_text), chunk_size)
]

print("Number of chunks:", len(chunks))

Number of chunks: 1353


## Step 3: Generate Embeddings

In [14]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\deves\OneDrive\Documents\Gen AI Assignments\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\deves\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

## Step 4: Build FAISS Index

In [16]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(embeddings)

In [17]:
print(index.ntotal)

1353


In [27]:
question = "What was total revenue in 2024 for the standalone business? And What are the key messages from the Founder/CEO/etc.?"

In [28]:
question_embedding = embedding_model.encode(
    [question]
)

distances, indices = index.search(
    question_embedding,
    k=3
)

In [29]:
context = ""

for idx in indices[0]:
    context += chunks[idx]
    context += "\n\n"

In [30]:
prompt = f"""
Answer the question using ONLY
the provided context.

Context:

{context}

Question:

{question}
"""

In [31]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

answer = response.choices[0].message.content

print(answer)

The total revenue for the standalone business in 2024 was ₹49,099.77 crore. 

Key messages from the Managing Director and Chief Executive Officer, Angshu Mallick, include:

1. Gratitude to stakeholders for their invaluable support.
2. Appreciation for shareholders' continued trust and support, which inspires the company to push boundaries and achieve greater heights.
3. Acknowledgment of employees' dedication, passion, and hard work as the driving force behind the company's success.
4. A commitment to navigate challenges and seize opportunities to ensure A WL Agri Business Limited remains a leader in the agri-business industry.
